# 第 3 周：深度学习、逻辑回归与反向传播

**对应主线**：李宏毅 ML 2022 的深度学习导论前置实践。  
**目标**：用一个原创二维二分类任务理解 logits、损失、梯度和参数更新。  
**先修**：第 2 周的矩阵乘法与训练循环；需要 `numpy`、`torch`。

完成后你应能解释：

- 线性层为什么输出 logits，而非先手动做阈值判断；
- `BCEWithLogitsLoss` 为何把 sigmoid 与交叉熵合在一起；
- `backward()` 产生什么，`step()` 改变什么。


In [ ]:
import numpy as np
import torch
from torch import nn

np.random.seed(7)
torch.manual_seed(7)


## 1. 从线性分数到概率

逻辑回归先计算 `logit = x @ w + b`，再通过 sigmoid 映射到 0–1。训练时使用 logits 更稳定。


In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


x_demo = np.array([[2.0, 1.0], [-1.0, -2.0]])
w_demo = np.array([1.5, 0.5])
b_demo = -0.25
logits_demo = x_demo @ w_demo + b_demo
probabilities_demo = sigmoid(logits_demo)

print("logits:", logits_demo)
print("probabilities:", probabilities_demo)


In [ ]:
assert logits_demo.shape == (2,)
assert np.all((probabilities_demo > 0) & (probabilities_demo < 1))
assert probabilities_demo[0] > 0.5
assert probabilities_demo[1] < 0.5
print("logit/sigmoid 测试通过 ✅")


## 2. 原创合成任务

标签规则是 `1.4*x1 - 0.8*x2 + 噪声 > 0`。模型不知道规则，只看到样本和标签。


In [ ]:
rng = np.random.default_rng(7)
features_np = rng.normal(size=(240, 2)).astype("float32")
noise = rng.normal(scale=0.25, size=240)
labels_np = (1.4 * features_np[:, 0] - 0.8 * features_np[:, 1] + noise > 0).astype("float32")

features = torch.tensor(features_np)
labels = torch.tensor(labels_np).reshape(-1, 1)

print(features.shape, labels.shape, labels.mean().item())


## 3. 先看一次梯度

梯度保存在参数的 `.grad` 中。`optimizer.step()` 根据这些梯度改变参数。


In [ ]:
model = nn.Linear(2, 1)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)

optimizer.zero_grad()
logits = model(features)
loss = loss_fn(logits, labels)
loss.backward()

weight_before = model.weight.detach().clone()
print("loss:", loss.item())
print("weight gradient:", model.weight.grad)

optimizer.step()
weight_after = model.weight.detach().clone()
print("parameter changed:", not torch.equal(weight_before, weight_after))


In [ ]:
assert model.weight.grad is not None
assert model.weight.grad.shape == model.weight.shape
assert not torch.equal(weight_before, weight_after)
print("反向传播测试通过 ✅")


## 4. 完整训练与评估


In [ ]:
torch.manual_seed(7)
model = nn.Linear(2, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)

for epoch in range(150):
    optimizer.zero_grad()
    logits = model(features)
    loss = loss_fn(logits, labels)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    probabilities = torch.sigmoid(model(features))
    predictions = (probabilities >= 0.5).float()
    accuracy = (predictions == labels).float().mean().item()

print(f"loss={loss.item():.4f}, accuracy={accuracy:.3f}")


In [ ]:
assert model(features).shape == (240, 1)
assert accuracy > 0.90
assert loss.item() < 0.30
print("逻辑回归训练测试通过 ✅")


## 小练习

1. 若 batch 是 `(64, 2)`，权重是 `(1, 2)`，PyTorch 线性层输出为何是 `(64, 1)`？
2. 把学习率改成 `20`，观察 loss；再恢复。用一句话说明发生了什么。
3. 不看上方代码，从空白重写训练循环。


## 常见错误

- 把 sigmoid 后的概率传给 `BCEWithLogitsLoss`：等于重复 sigmoid。
- 标签 shape 是 `(batch,)`、输出是 `(batch, 1)`：损失函数会报 shape 不匹配。
- 用准确率反向传播：阈值操作不可导，训练应使用连续的损失。
- 用训练集准确率代替泛化能力：下周开始固定划分验证集。


## 扩展与出口测验

扩展：新增第三个特征，但让它完全是噪声，比较学到的三个权重。  
出口测验：闭卷解释 `logit → loss → gradient → parameter update`，并指出 `backward` 与 `step` 的差别。

**通过条件**：全部断言通过，准确率超过 90%，且能解释一次错误学习率实验。
